# Notebook 04 -- Higher-Order Synthetic (Figures 4, 17, 18)

This notebook generates pseudofractal simplicial complexes and runs
higher-order Hodge Laplacian renormalization to produce spectral and
harmonic-degree diagnostics.

**Figures produced:**
- **Figure 4:** 2x3 panel -- spectral properties (entropic susceptibility,
  entropy, spectral dimension) and harmonic degree curves for the
  pseudofractal d2 complex under Hodge, Bochner, and cross-order Laplacians.
- **Figure 17:** Compression curves for pseudofractal d2.
- **Figure 18:** Compression curves for pseudofractal d3.

In [ ]:
%matplotlib inline
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from harmonic_morphisms import g_len, higher_order_H_CF_curves, fbc
from harmonic_morphisms.simplicial import (
    pseudofractal_d2, pseudofractal_d3, laplacians,
    XO_laplacian, compute_entropic_C, compute_spectral_d, plot_complex,
)
from harmonic_morphisms.higher_order import higher_order_renormalization

SMALL_SIZE = 10; MEDIUM_SIZE = 12; BIGGER_SIZE = 13
plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=SMALL_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

## Section 1: Generate Pseudofractal d2 (3 iterations)

In [ ]:
sc = pseudofractal_d2(3)
print(f"Pseudofractal d2 (3 iter): {sc['n0']} nodes, {sc['n1']} edges, {sc['n2']} faces")

L0, L1, L2, L3, L4, node_dict, edge_dict, face_dict, tet_dict = laplacians(sc)

# Bochner decomposition of L2
B2, F2 = fbc(L2)

# Cross-order Laplacian (faces diffusing via shared edges)
X2 = XO_laplacian(sc, 2, 1)

# Eigendecompositions
e2, ev2 = np.linalg.eigh(L2)
eb2, evb2 = np.linalg.eigh(B2)
ex2, evx2 = np.linalg.eigh(X2)

print(f"L2 shape: {L2.shape}, B2 shape: {B2.shape}, X2 shape: {X2.shape}")

## Section 2: Figure 4 -- Spectral properties and harmonic degree curves (2x3 panel)

In [ ]:
# Spectral quantities for each operator
C_h, tau_h, S_h = compute_entropic_C(e2, -2, 5, 1000)
C_b, tau_b, S_b = compute_entropic_C(eb2, -2, 5, 1000)
C_x, tau_x, S_x = compute_entropic_C(ex2, -2, 5, 1000)

dS_h, tau_dS_h = compute_spectral_d(e2, -2, 5, 1000)
dS_b, tau_dS_b = compute_spectral_d(eb2, -2, 5, 1000)
dS_x, tau_dS_x = compute_spectral_d(ex2, -2, 5, 1000)

In [ ]:
# Harmonic degree curves for each operator
print("Running higher_order_H_CF_curves with Hodge L2...")
Ag_h, g_h, DEG_H_h, M_DEG_H_h, STD_H_h, AV_H_h, STD_V_H_h, NOT_H_h, \
    DEG_CF_h, M_DEG_CF_h, STD_CF_h, AV_CF_h, STD_V_CF_h, NOT_CF_h, \
    gV_h, t_span_h = higher_order_H_CF_curves(sc, dim=2, Lap=L2, n=100)

print("Running higher_order_H_CF_curves with Bochner B2...")
Ag_b, g_b, DEG_H_b, M_DEG_H_b, STD_H_b, AV_H_b, STD_V_H_b, NOT_H_b, \
    DEG_CF_b, M_DEG_CF_b, STD_CF_b, AV_CF_b, STD_V_CF_b, NOT_CF_b, \
    gV_b, t_span_b = higher_order_H_CF_curves(sc, dim=2, Lap=B2, n=100)

print("Running higher_order_H_CF_curves with XO X2...")
Ag_x, g_x, DEG_H_x, M_DEG_H_x, STD_H_x, AV_H_x, STD_V_H_x, NOT_H_x, \
    DEG_CF_x, M_DEG_CF_x, STD_CF_x, AV_CF_x, STD_V_CF_x, NOT_CF_x, \
    gV_x, t_span_x = higher_order_H_CF_curves(sc, dim=2, Lap=X2, n=100)

print("Done.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5))

# -- Row 0: Spectral properties --
# Entropic susceptibility
ax = axes[0, 0]
ax.loglog(tau_h, C_h, color="red", linewidth=3.5, label="Hodge-2")
ax.loglog(tau_b, C_b, color="#FF8605", linewidth=3.5, label="Bochner-2")
ax.loglog(tau_x, C_x, color="#0291F7", linewidth=3.5, label="XO-2")
ax.set_xlabel("t")
ax.set_ylabel("Entropic susceptibility")
ax.legend()

# Entropy
ax = axes[0, 1]
tau_S = np.logspace(-2, 5, 1000)
ax.loglog(tau_S, S_h, color="red", linewidth=3.5, label="Hodge-2")
ax.loglog(tau_S, S_b, color="#FF8605", linewidth=3.5, label="Bochner-2")
ax.loglog(tau_S, S_x, color="#0291F7", linewidth=3.5, label="XO-2")
ax.set_xlabel("t")
ax.set_ylabel("Entropy")
ax.legend()

# Spectral dimension
ax = axes[0, 2]
ax.loglog(tau_dS_h, dS_h, color="red", linewidth=3.5, label="Hodge-2")
ax.loglog(tau_dS_b, dS_b, color="#FF8605", linewidth=3.5, label="Bochner-2")
ax.loglog(tau_dS_x, dS_x, color="#0291F7", linewidth=3.5, label="XO-2")
ax.set_xlabel("t")
ax.set_ylabel("Spectral dimension")
ax.legend()

# -- Row 1: Harmonic degree curves vs time --
operators = [
    ("Hodge L2", t_span_h, M_DEG_H_h, M_DEG_CF_h, STD_H_h),
    ("Bochner B2", t_span_b, M_DEG_H_b, M_DEG_CF_b, STD_H_b),
    ("XO X2", t_span_x, M_DEG_H_x, M_DEG_CF_x, STD_H_x),
]

for col, (name, t_span, m_h, m_cf, std_h) in enumerate(operators):
    ax = axes[1, col]
    ax.plot(t_span, m_h, color="blue", linewidth=2.5, label="H Mod.")
    ax.plot(t_span, m_cf, color="green", linewidth=2.5, label="CF Mod.")
    ax.plot(t_span, std_h, color="black", linewidth=2.5, label="STD H")
    ax.set_xlabel("t")
    ax.set_ylabel("Harmonic degree")
    ax.set_ylim([-0.05, 1.2])
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.savefig("fig04_pf2_panel.pdf", bbox_inches="tight")
plt.show()
print("Saved fig04_pf2_panel.pdf")

## Section 3: Figure 17 -- Compression curves for pseudofractal d2

In [ ]:
n_faces = sc["n2"]

comp_h = 1 - np.array(g_len(g_h)) / n_faces
comp_b = 1 - np.array(g_len(g_b)) / n_faces
comp_x = 1 - np.array(g_len(g_x)) / n_faces

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name, comp, m_h, m_cf, std_h in [
    (axes[0], "Hodge L2", comp_h, M_DEG_H_h, M_DEG_CF_h, STD_H_h),
    (axes[1], "Bochner B2", comp_b, M_DEG_H_b, M_DEG_CF_b, STD_H_b),
    (axes[2], "XO X2", comp_x, M_DEG_H_x, M_DEG_CF_x, STD_H_x),
]:
    ax.plot(comp, m_h, "ob--", linewidth=1.5, markersize=4, label="H Mod.")
    ax.plot(comp, m_cf, "og--", linewidth=1.5, markersize=4, label="CF Mod.")
    ax.plot(comp, std_h, "ok--", linewidth=1.5, markersize=4, label="STD H")
    ax.set_xlabel("Compression")
    ax.set_ylabel("Harmonic degree")
    ax.set_ylim([-0.1, 1.2])
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Pseudofractal d2 -- compression curves")
plt.tight_layout()
plt.savefig("fig17_pf2_compression.pdf", bbox_inches="tight")
plt.show()
print("Saved fig17_pf2_compression.pdf")

## Section 4: Figure 18 -- Pseudofractal d3 (2 iterations)

In [ ]:
sc3 = pseudofractal_d3(2)
print(f"Pseudofractal d3 (2 iter): {sc3['n0']} nodes, {sc3['n1']} edges, "
      f"{sc3['n2']} faces, {sc3['n3']} tetrahedra")

L0_3, L1_3, L2_3, L3_3, L4_3, node_dict_3, edge_dict_3, face_dict_3, tet_dict_3 = laplacians(sc3)

# Bochner decomposition of L2 for d3 complex
B2_3, F2_3 = fbc(L2_3)

# Cross-order Laplacian for d3 complex
X2_3 = XO_laplacian(sc3, 2, 1)

In [ ]:
# Harmonic degree curves for d3 complex
print("Running higher_order_H_CF_curves on d3 with Hodge L2...")
Ag_h3, g_h3, DEG_H_h3, M_DEG_H_h3, STD_H_h3, AV_H_h3, STD_V_H_h3, NOT_H_h3, \
    DEG_CF_h3, M_DEG_CF_h3, STD_CF_h3, AV_CF_h3, STD_V_CF_h3, NOT_CF_h3, \
    gV_h3, t_span_h3 = higher_order_H_CF_curves(sc3, dim=2, Lap=L2_3, n=100)

print("Running higher_order_H_CF_curves on d3 with Bochner B2...")
Ag_b3, g_b3, DEG_H_b3, M_DEG_H_b3, STD_H_b3, AV_H_b3, STD_V_H_b3, NOT_H_b3, \
    DEG_CF_b3, M_DEG_CF_b3, STD_CF_b3, AV_CF_b3, STD_V_CF_b3, NOT_CF_b3, \
    gV_b3, t_span_b3 = higher_order_H_CF_curves(sc3, dim=2, Lap=B2_3, n=100)

print("Running higher_order_H_CF_curves on d3 with XO X2...")
Ag_x3, g_x3, DEG_H_x3, M_DEG_H_x3, STD_H_x3, AV_H_x3, STD_V_H_x3, NOT_H_x3, \
    DEG_CF_x3, M_DEG_CF_x3, STD_CF_x3, AV_CF_x3, STD_V_CF_x3, NOT_CF_x3, \
    gV_x3, t_span_x3 = higher_order_H_CF_curves(sc3, dim=2, Lap=X2_3, n=100)

print("Done.")

In [ ]:
n_faces_3 = sc3["n2"]

comp_h3 = 1 - np.array(g_len(g_h3)) / n_faces_3
comp_b3 = 1 - np.array(g_len(g_b3)) / n_faces_3
comp_x3 = 1 - np.array(g_len(g_x3)) / n_faces_3

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name, comp, m_h, m_cf, std_h in [
    (axes[0], "Hodge L2", comp_h3, M_DEG_H_h3, M_DEG_CF_h3, STD_H_h3),
    (axes[1], "Bochner B2", comp_b3, M_DEG_H_b3, M_DEG_CF_b3, STD_H_b3),
    (axes[2], "XO X2", comp_x3, M_DEG_H_x3, M_DEG_CF_x3, STD_H_x3),
]:
    ax.plot(comp, m_h, "ob--", linewidth=1.5, markersize=4, label="H Mod.")
    ax.plot(comp, m_cf, "og--", linewidth=1.5, markersize=4, label="CF Mod.")
    ax.plot(comp, std_h, "ok--", linewidth=1.5, markersize=4, label="STD H")
    ax.set_xlabel("Compression")
    ax.set_ylabel("Harmonic degree")
    ax.set_ylim([-0.1, 1.2])
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Pseudofractal d3 -- compression curves")
plt.tight_layout()
plt.savefig("fig18_pf3_compression.pdf", bbox_inches="tight")
plt.show()
print("Saved fig18_pf3_compression.pdf")